In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import numpy as np

spark = SparkSession.builder.getOrCreate()

folder = "../prj-p-ai-ailabs/input/mliu30/data/cdss_us_sbs"

In [30]:
header_df = pd.read_csv(f"{folder}/us_sbs_gen9_cdss_hdr_xgb.csv", header=None)
column_names = header_df.iloc[0].tolist()
print(len(column_names))
# print(column_names)

310


In [23]:
header_df_1 = pd.read_csv(f"{folder}/us_sbs_cdss_gen9_header_upd_no_preproc.csv", header=None)
column_names_1 = header_df_1.iloc[0].tolist()
print(len(column_names_1))
print([(i, column_names[i], column_names_1[i]) for i in range(len(column_names_1)) if column_names[i] != column_names_1[i]])
# 'RLINWVAR' is not in the variable list

310
[(309, 'RLIIQ180', 'RLINWVAR')]


In [29]:
# label
print(header_df.iloc[1].value_counts())
label_idx = np.arange(len(column_names))[[l=="L" for l in header_df.iloc[1].tolist()]][0]
print(label_idx)
label = column_names[label_idx]
print(label)

1
N    308
K      1
L      1
Name: count, dtype: int64
273
DEFAULT


In [33]:
# unused features
unused = ["REFACTRI", "EBINEXRT", "EDFSCR1", "FCSTDBRT", "HILMSENR", "ONS30HRD"]
print([v in column_names for v in unused])

[True, True, True, True, True, True]


In [48]:
# dev (A)
data = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "true")
    .csv(f"{folder}/cdss_dev_csv.csv")
    .toDF(*column_names)
)

data = data.drop(*unused)

print(len(data.columns))

# data.show(2, truncate=False)
data.printSchema()

data.count()

306
root
 |-- CUST_ID_NEW: string (nullable = true)
 |-- AC99DMCG: double (nullable = true)
 |-- AC99DVCG: double (nullable = true)
 |-- AIR_D30: double (nullable = true)
 |-- AULNPYMT: double (nullable = true)
 |-- AUTBALAM: double (nullable = true)
 |-- CEMSAUTL: integer (nullable = true)
 |-- ATOPYMTI: integer (nullable = true)
 |-- AUTPAY12: double (nullable = true)
 |-- CEAULCNT: integer (nullable = true)
 |-- BDGLBLEN: integer (nullable = true)
 |-- BESTTNUR: integer (nullable = true)
 |-- CUSBERAU: double (nullable = true)
 |-- BFFINBAL: double (nullable = true)
 |-- CUSBEFNQ: integer (nullable = true)
 |-- BFLNDTEN: integer (nullable = true)
 |-- BFLNTEN: integer (nullable = true)
 |-- CUSB100U: integer (nullable = true)
 |-- BFPMTAM: double (nullable = true)
 |-- BFSBESCR: integer (nullable = true)
 |-- CUSBETLB: integer (nullable = true)
 |-- CASUTLRL: double (nullable = true)
 |-- CEMXBRAM: integer (nullable = true)
 |-- CDDACT1Y: integer (nullable = true)
 |-- CESRCIND: dou

20026265

In [42]:
# from pyspark.sql import functions as F

# grouped = (
#     data
#     .withColumn("month", F.split(F.col("CUST_ID_NEW"), "_").getItem(1))
#     .groupBy("month")
#     .count()
# )

# grouped.show()

In [41]:
from pyspark.sql import functions as F

data = data.withColumn(
    "month_raw",
    F.element_at(F.split(F.col("CUST_ID_NEW"), "_"), 2)
)

# Normalize month to yyyy-MM
data = data.withColumn(
    "month",
    F.when(
        F.to_date(F.col("month_raw"), "yyyyMM").isNotNull(),# 202304
        F.date_format(F.to_date(F.col("month_raw"), "yyyyMM"), "yyyy-MM")
    ).when(
        F.to_date(F.col("month_raw"), "MMMyy").isNotNull(), # jan23
        F.date_format(F.to_date(F.col("month_raw"), "MMMyy"), "yyyy-MM")
    )
)

# Check what was extracted / normalized
data.select("CUST_ID_NEW", "month_raw", "month").show(20, truncate=False)

# Group by month
monthly = (
    data.groupBy("month")
        .count()
        .orderBy("month")
)

monthly.show()

+--------------------------+---------+-------+
|CUST_ID_NEW               |month_raw|month  |
+--------------------------+---------+-------+
|773292740014_202404_6_SPD |202404   |2024-04|
|43485126016_202404_6_SPD  |202404   |2024-04|
|814134164016_202311_6_SPD |202311   |2023-11|
|54944651016_202304_18_SPD |202304   |2023-04|
|30917667015_202405_6_SPD  |202405   |2024-05|
|34740054011_202301_18_SPD |202301   |2023-01|
|298015644011_202403_6_SPD |202403   |2024-03|
|804387089011_202405_6_SPD |202405   |2024-05|
|17131629010_202404_6_SPD  |202404   |2024-04|
|593088262015_202404_6_SPD |202404   |2024-04|
|683750881016_202304_18_SPD|202304   |2023-04|
|681362786016_202405_6_SPD |202405   |2024-05|
|129092659012_202401_6_SPD |202401   |2024-01|
|9119386015_202211_18_SPD  |202211   |2022-11|
|280263449014_202401_6_SPD |202401   |2024-01|
|761546285015_202304_18_SPD|202304   |2023-04|
|313454676017_202211_18_SPD|202211   |2022-11|
|464227820405_202311_6_SPD |202311   |2023-11|
|865464716013

+-------+-------+
|  month|  count|
+-------+-------+
|2022-11|1821588|
|2023-01|1859064|
|2023-03|1884387|
|2023-04|1893455|
|2023-05|1928522|
|2023-11|2077547|
|2024-01|2119386|
|2024-03|2137884|
|2024-04|2146618|
|2024-05|2157814|
+-------+-------+



In [ ]:
selected_month = "2023-05"

In [ ]:
## 1-month dev sample for model training
sample_dev = (
    data.filter(F.col("month") == selected_month)
        .orderBy(F.rand(seed=42))
        .cache()
)

sample_dev.agg(
    F.count("*").alias("sample_count"),
    F.avg(F.col("DEFAULT").cast("double")).alias("event_rate")
).show()

sample_dev.write.mode("overwrite").option("header", "true").csv(
    f"{folder}/samples/sample_dev"
)


In [ ]:
## dev sample for feature engineering
n_sample = 20000

label_counts = {
    row["DEFAULT"]: row["count"]
    for row in data.filter(F.col("month") == selected_month)
                   .groupBy("DEFAULT")
                   .count()
                   .collect()
}

per_label = n_sample / len(label_counts)
fractions = {
    label_value: min(per_label / count, 1.0)
    for label_value, count in label_counts.items()
}

sample_dev = (
    data.filter(F.col("month") == selected_month)
        .sampleBy("DEFAULT", fractions=fractions, seed=42)
        .cache()
)

sample_dev.agg(
    F.count("*").alias("sample_count"),
    F.avg(F.col("DEFAULT").cast("double")).alias("event_rate")
).show()

sample_dev.write.mode("overwrite").option("header", "true").csv(
    f"{folder}/samples/sample_dev_fe"
)

In [49]:
# OOS (B)
data_oos = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "true")
    .csv(f"{folder}/cdss_oos_csv.csv")
    .toDF(*column_names)
)

data_oos = data_oos.drop(*unused)

# data.show(2, truncate=False)
# data_oos.printSchema()

data_oos.count()

root
 |-- CUST_ID_NEW: string (nullable = true)
 |-- AC99DMCG: double (nullable = true)
 |-- AC99DVCG: double (nullable = true)
 |-- AIR_D30: double (nullable = true)
 |-- AULNPYMT: double (nullable = true)
 |-- AUTBALAM: double (nullable = true)
 |-- CEMSAUTL: integer (nullable = true)
 |-- ATOPYMTI: integer (nullable = true)
 |-- AUTPAY12: double (nullable = true)
 |-- CEAULCNT: integer (nullable = true)
 |-- BDGLBLEN: integer (nullable = true)
 |-- BESTTNUR: integer (nullable = true)
 |-- CUSBERAU: double (nullable = true)
 |-- BFFINBAL: double (nullable = true)
 |-- CUSBEFNQ: integer (nullable = true)
 |-- BFLNDTEN: integer (nullable = true)
 |-- BFLNTEN: integer (nullable = true)
 |-- CUSB100U: integer (nullable = true)
 |-- BFPMTAM: double (nullable = true)
 |-- BFSBESCR: integer (nullable = true)
 |-- CUSBETLB: integer (nullable = true)
 |-- CASUTLRL: double (nullable = true)
 |-- CEMXBRAM: integer (nullable = true)
 |-- CDDACT1Y: integer (nullable = true)
 |-- CESRCIND: double 

3942497

In [51]:
grouped_oos = (
    data_oos
    .withColumn("month", F.split(F.col("CUST_ID_NEW"), "_").getItem(1))
    .groupBy("month")
    .count()
)

grouped_oos.show()

+------+------+
| month| count|
+------+------+
|202304|798794|
|202303|789674|
|202305|808158|
|202301|779698|
|202211|766173|
+------+------+



In [50]:
from pyspark.sql import functions as F

data_oos = data_oos.withColumn(
    "month_raw",
    F.element_at(F.split(F.col("CUST_ID_NEW"), "_"), 2)
)

# Normalize month to yyyy-MM
data_oos = data_oos.withColumn(
    "month",
    F.when(
        F.to_date(F.col("month_raw"), "yyyyMM").isNotNull(),# 202304
        F.date_format(F.to_date(F.col("month_raw"), "yyyyMM"), "yyyy-MM")
    ).when(
        F.to_date(F.col("month_raw"), "MMMyy").isNotNull(), # jan23
        F.date_format(F.to_date(F.col("month_raw"), "MMMyy"), "yyyy-MM")
    )
)

# Check what was extracted / normalized
data_oos.select("CUST_ID_NEW", "month_raw", "month").show(20, truncate=False)

# Group by month
monthly_oos = (
    data_oos.groupBy("month")
        .count()
        .orderBy("month")
)

monthly_oos.show()

+--------------------------+---------+-------+
|CUST_ID_NEW               |month_raw|month  |
+--------------------------+---------+-------+
|572622479014_202211_18_SPD|202211   |2022-11|
|20449635010_202301_18_SPD |202301   |2023-01|
|48092607019_202305_18_SPD |202305   |2023-05|
|3651245010_202211_18_SPD  |202211   |2022-11|
|654006775018_202305_18_SPD|202305   |2023-05|
|48199366013_202304_18_CSL |202304   |2023-04|
|664729242012_202304_18_SPD|202304   |2023-04|
|120062359013_202303_18_SPD|202303   |2023-03|
|366654143408_202301_18_SPD|202301   |2023-01|
|147967311017_202304_18_SPD|202304   |2023-04|
|9617188013_202303_18_SPD  |202303   |2023-03|
|586776242012_202301_18_SPD|202301   |2023-01|
|28475925011_202301_18_SPD |202301   |2023-01|
|9662632019_202304_18_SPD  |202304   |2023-04|
|470685474405_202211_18_SPD|202211   |2022-11|
|51012608018_202305_18_SPD |202305   |2023-05|
|281553585018_202305_18_SPD|202305   |2023-05|
|623630703014_202301_18_SPD|202301   |2023-01|
|40059774010_

+-------+------+
|  month| count|
+-------+------+
|2022-11|766173|
|2023-01|779698|
|2023-03|789674|
|2023-04|798794|
|2023-05|808158|
+-------+------+



In [ ]:
## 1-month oos sample for model validation
sample_oos = (
    data_oos.filter(F.col("month") == selected_month)
        .orderBy(F.rand(seed=42))
        .cache()
)

sample_oos.agg(
    F.count("*").alias("sample_count"),
    F.avg(F.col("DEFAULT").cast("double")).alias("event_rate")
).show()

sample_oos.write.mode("overwrite").option("header", "true").csv(
    f"{folder}/samples/sample_oos"
)


In [ ]:
## oos sample for feature engineering
n_sample = 2000

label_counts = {
    row["DEFAULT"]: row["count"]
    for row in data_oos.filter(F.col("month") == selected_month)
                       .groupBy("DEFAULT").count().collect()
}

per_label = n_sample / len(label_counts)
fractions = {
    label_value: min(per_label / count, 1.0)
    for label_value, count in label_counts.items()
}

sample_oos_fe = (
    data_oos.filter(F.col("month") == selected_month)
        .sampleBy("DEFAULT", fractions=fractions, seed=42)
        .cache()
)

sample_oos_fe.agg(
    F.count("*").alias("sample_count"),
    F.avg(F.col("DEFAULT").cast("double")).alias("event_rate")
).show()


sample_oos_fe.write.mode("overwrite").option("header", "true").csv(
    f"{folder}/samples/sample_oos_fe"
)